# Colab run v4: corrected source data, category-level clothing evaluation, retrained adapter
Changes vs v3 (all in `src/`): ASTE text is **detokenised**, duplicate (aspect, sentiment) gold entries are **merged**, dev/test contain **aspect-free sentences**, relaxed matching is **token-level**, and the clothing evaluation is **category-level** (gold labels are mostly categories, not spans; both sides map to `configs/canonical_aspects.json`; span-level literal-gold score and net-sentiment agreement are reported too; text comes from the pipeline data). Because training data changed, the fine-tuned adapter is **retrained** (new path `absa-qlora-v4`). All old results are invalid and archived.

Upload `capstone-absa-colab-v4.zip` to `/content`, set **Runtime > T4 GPU**, run in order. Training takes roughly an hour; it resumes from Drive checkpoints if the runtime disconnects.

In [1]:
import os, zipfile
from google.colab import drive

# 1. Mount Google Drive if not already mounted
if not os.path.isdir("/content/drive"):
    drive.mount("/content/drive")

# 2. Check for the source zip and extract it FIRST
ZIP, PROJECT = "/content/capstone-absa-colab-v4.zip", "/content/capstone-absa-colab"
if not os.path.exists(ZIP):
    from google.colab import files; files.upload()

os.makedirs(PROJECT, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(PROJECT)

# 3. Assert project files exist now that extraction is complete
assert os.path.isdir(f"{PROJECT}/src"), "Project not found in /content: extraction failed."

# 4. Change directory, install dependencies, and download spaCy models
%cd $PROJECT
!nvidia-smi -L
!pip install -q -r requirements.txt pytest
!pip freeze > pip_freeze.txt
!python -m spacy download en_core_web_sm -q

# 5. Define global variables
MODEL = "unsloth/Llama-3.2-3B-Instruct"
ADAPTER = "/content/drive/MyDrive/absa-qlora-v4"   # NEW adapter, trained on the corrected data

/content/capstone-absa-colab
GPU 0: Tesla T4 (UUID: GPU-4969f58a-71c1-0404-c7d8-ffcf85515562)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
# Rebuild data with the v4 preprocessing, then unit + data-invariant tests
import os
PROJECT, MODEL, ADAPTER = "/content/capstone-absa-colab", "unsloth/Llama-3.2-3B-Instruct", "/content/drive/MyDrive/absa-qlora-v4"
assert os.path.isdir(f"{PROJECT}/src"), "Project not found in /content: run the setup cell first (it unzips capstone-absa-colab-v4.zip)"
%cd $PROJECT
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
if not os.path.exists("data/raw/amazon_clothing_sample.jsonl"):
    !python scripts/fetch_data.py amazon --n 5000 --out data/raw/amazon_clothing_sample.jsonl
!python -m src.prepare_data && python -m src.build_sft_data && python -m src.prepare_clothing
!python -m pytest -q tests

/content/capstone-absa-colab
train sentences removed (duplicates / leakage): 10
cross-split duplicate sentences train/dev: 0
cross-split duplicate sentences train/test: 0
cross-split duplicate sentences dev/test: 0
train within-split duplicate sentences: 0
train sentences 2703 aspects 3300 {'positive': 2145, 'negative': 880, 'neutral': 275} no-aspect 541
dev within-split duplicate sentences: 0
dev sentences 662 aspects 794 {'positive': 489, 'negative': 222, 'neutral': 83} no-aspect 133
test within-split duplicate sentences: 0
test sentences 1025 aspects 1311 {'positive': 953, 'negative': 241, 'neutral': 117} no-aspect 205
train 2703
dev 662
{
  "raw_reviews": 5000,
  "identifier_fields_found_and_dropped": [],
  "after_cleaning_dedup": 5000,
  "sentences_total": 25206,
  "sentences_per_review_mean": 5.04,
  "sentence_length_words_mean": 11.63
}
....................                                                     [100%]
20 passed in 4.49s


In [2]:
# Archive any OLD (v3) results from Drive, then start clean
import os
PROJECT, MODEL, ADAPTER = "/content/capstone-absa-colab", "unsloth/Llama-3.2-3B-Instruct", "/content/drive/MyDrive/absa-qlora-v4"
assert os.path.isdir(f"{PROJECT}/src"), "Project not found in /content: run the setup cell first (it unzips capstone-absa-colab-v4.zip)"
%cd $PROJECT
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
import shutil, os, time
old = f"/content/drive/MyDrive/absa-results-old-{int(time.time())}"
if os.path.isdir("/content/drive/MyDrive/absa-results"): shutil.copytree("/content/drive/MyDrive/absa-results", old); print("old results archived to", old)
for f in ("baseline_dev","baseline_test","prompted_dev","prompted_test","finetuned_dev","finetuned_test","clothing_baseline","clothing_baseline_lr","clothing_prompted","clothing_finetuned"):
    p=f"results/{f}.json"
    if os.path.exists(p): os.remove(p)

/content/capstone-absa-colab
old results archived to /content/drive/MyDrive/absa-results-old-1795670067


In [3]:
# Baselines on the corrected source dev/test (these are the in-domain reference for the drop)
import os
PROJECT, MODEL, ADAPTER = "/content/capstone-absa-colab", "unsloth/Llama-3.2-3B-Instruct", "/content/drive/MyDrive/absa-qlora-v4"
assert os.path.isdir(f"{PROJECT}/src"), "Project not found in /content: run the setup cell first (it unzips capstone-absa-colab-v4.zip)"
%cd $PROJECT
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
!python -m src.baseline_spacy --split dev
!python -m src.baseline_spacy --split test

/content/capstone-absa-colab
spacy_lexicon aspect F1 exact 0.402 relaxed 0.457 | pair F1 exact 0.368 relaxed 0.420
spacy_tfidf_lr aspect F1 exact 0.402 relaxed 0.457 | pair F1 exact 0.332 relaxed 0.381
TF-IDF/LR sentiment acc (gold aspects): 0.724
spacy_lexicon aspect F1 exact 0.436 relaxed 0.513 | pair F1 exact 0.416 relaxed 0.487
spacy_tfidf_lr aspect F1 exact 0.436 relaxed 0.513 | pair F1 exact 0.384 relaxed 0.450
TF-IDF/LR sentiment acc (gold aspects): 0.777


In [4]:
# QLoRA fine-tuning on the corrected data (best eval-loss epoch is kept; resumes from Drive checkpoints)
# If you hit CUDA out-of-memory, rerun with --batch 2
import os
PROJECT, MODEL, ADAPTER = "/content/capstone-absa-colab", "unsloth/Llama-3.2-3B-Instruct", "/content/drive/MyDrive/absa-qlora-v4"
assert os.path.isdir(f"{PROJECT}/src"), "Project not found in /content: run the setup cell first (it unzips capstone-absa-colab-v4.zip)"
%cd $PROJECT
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
!python -m src.finetune_qlora --model $MODEL --out $ADAPTER --epochs 3 --batch 4
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "training did not finish"

/content/capstone-absa-colab
compute dtype: torch.float16
Loading weights: 100% 254/254 [00:20<00:00, 12.47it/s]
trainable params: 9,175,040 || all params: 3,221,924,864 || trainable%: 0.2848
Generating train split: 2703 examples [00:00, 159210.26 examples/s]
Generating dev split: 662 examples [00:00, 153966.36 examples/s]
Map: 100% 2703/2703 [00:01<00:00, 1993.34 examples/s]
Map: 100% 662/662 [00:00<00:00, 2069.67 examples/s]
  0% 0/507 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
{'loss': '0.03428', 'grad_norm': '0.2524', 'learnin

In [5]:
# DIAGNOSTIC (2 min): raw outputs of the NEW adapter on real clothing sentences (v3 returned [] for ~all of them)
import os
PROJECT, MODEL, ADAPTER = "/content/capstone-absa-colab", "unsloth/Llama-3.2-3B-Instruct", "/content/drive/MyDrive/absa-qlora-v4"
assert os.path.isdir(f"{PROJECT}/src"), "Project not found in /content: run the setup cell first (it unzips capstone-absa-colab-v4.zip)"
%cd $PROJECT
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
import sys; sys.path.insert(0, ".")
from src.common import read_jsonl
from src.llm_prompted import hf_generator, extract_with_retry
rows = read_jsonl("data/processed/clothing_reviews.jsonl")[:12]
sents = [s for r in rows for s in r["sentences"]][:40]
gen = hf_generator(MODEL, ADAPTER)
empty = 0
for s in sents:
    r = extract_with_retry(s, gen, few_shot=False)
    empty += (r["ok"] and not r["items"])
    print(repr(s[:110]), "->", r["raw"].strip()[:140])
print(f"\nempty predictions: {empty}/{len(sents)}")
del gen

[review sentences removed from this output before publishing; raw review text is not committed]
empty predictions: 24/40


In [ ]:
# Prompted LLM, source dev + test
import os
PROJECT, MODEL, ADAPTER = "/content/capstone-absa-colab", "unsloth/Llama-3.2-3B-Instruct", "/content/drive/MyDrive/absa-qlora-v4"
assert os.path.isdir(f"{PROJECT}/src"), "Project not found in /content: run the setup cell first (it unzips capstone-absa-colab-v4.zip)"
%cd $PROJECT
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
for split in ("dev", "test"):
    !python -m src.llm_prompted --split $split --model $MODEL

In [6]:
# Fine-tuned LLM, source dev + test
import os
PROJECT, MODEL, ADAPTER = "/content/capstone-absa-colab", "unsloth/Llama-3.2-3B-Instruct", "/content/drive/MyDrive/absa-qlora-v4"
assert os.path.isdir(f"{PROJECT}/src"), "Project not found in /content: run the setup cell first (it unzips capstone-absa-colab-v4.zip)"
%cd $PROJECT
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
for split in ("dev", "test"):
    !python -m src.llm_prompted --split $split --model $MODEL --adapter $ADAPTER

/content/capstone-absa-colab
Loading weights: 100% 254/254 [00:17<00:00, 14.40it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
{
  "method": "finetuned",
  "n": 662,
  "json_validity_rate": 0.9939577039274925,
  "seconds_per_review": 4.019724953930904,
  "metrics_strict_all": {
    "aspect_exact": {
      "precision": 0.8309114927344782,
      "recall": 0.792191435768262,
      "f1": 0.8110896196002578
    },
    "aspect_relaxed": {
      "precision": 0.8863936591809776,
      "recall": 0.845088161209068,
      "f1": 0.8652482269503546
    },
    "pair_exact": {
      "precision": 0.7516512549537648,
      "recall": 

In [7]:
# Clothing evaluation, all methods, then comparison
import os
PROJECT, MODEL, ADAPTER = "/content/capstone-absa-colab", "unsloth/Llama-3.2-3B-Instruct", "/content/drive/MyDrive/absa-qlora-v4"
assert os.path.isdir(f"{PROJECT}/src"), "Project not found in /content: run the setup cell first (it unzips capstone-absa-colab-v4.zip)"
%cd $PROJECT
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
for m in ("baseline", "baseline_lr"):
    !python -m src.eval_clothing --method $m --model $MODEL
!python -m src.eval_clothing --method prompted --model $MODEL
!python -m src.eval_clothing --method finetuned --model $MODEL --adapter $ADAPTER
!python -m src.compare

/content/capstone-absa-colab
gold reviews: 300
baseline on clothing | scored 300/300 | CATEGORY aspect F1 0.204 pair F1 0.176 | SPAN (literal gold) pair F1 0.176 | 0.02s/review
  net-sentiment agreement over 10 categories: Pearson 0.027201145715754334 Spearman -0.043217287550303775
  span drop aspect_exact source 0.436 -> clothing 0.214 (abs 0.222, rel 50.9%)
  span drop pair_exact source 0.416 -> clothing 0.176 (abs 0.240, rel 57.6%)
gold reviews: 300
baseline_lr on clothing | scored 300/300 | CATEGORY aspect F1 0.204 pair F1 0.158 | SPAN (literal gold) pair F1 0.160 | 0.02s/review
  net-sentiment agreement over 10 categories: Pearson -0.14464004075242937 Spearman -0.1533857797849622
  span drop aspect_exact source 0.436 -> clothing 0.214 (abs 0.222, rel 50.9%)
  span drop pair_exact source 0.384 -> clothing 0.160 (abs 0.224, rel 58.3%)
gold reviews: 300
Loading weights: 100% 254/254 [00:21<00:00, 11.91it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer T

In [8]:
# Aggregation for the report view with the LLM extractors
import os
PROJECT, MODEL, ADAPTER = "/content/capstone-absa-colab", "unsloth/Llama-3.2-3B-Instruct", "/content/drive/MyDrive/absa-qlora-v4"
assert os.path.isdir(f"{PROJECT}/src"), "Project not found in /content: run the setup cell first (it unzips capstone-absa-colab-v4.zip)"
%cd $PROJECT
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
!python -m src.aggregate --method finetuned --adapter $ADAPTER --model $MODEL --limit 300 --min_mentions 5
!python -m src.aggregate --method prompted --model $MODEL --limit 300 --min_mentions 5

/content/capstone-absa-colab
Loading weights: 100% 254/254 [00:21<00:00, 11.93it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
300 reviews, 1637 sentences, 1635 valid extractions -> /content/capstone-absa-colab/results/aggregate_clothing_finetuned.csv
sentiment   negative  neutral  positive  mentions  net_sentiment
aspect                                                          
material          22        5        35        62           0.21
fit                7        4        43        54           0.67
style              4        2        43        49           0.80
color              7        3        33        

In [9]:
# Save everything
import os
PROJECT, MODEL, ADAPTER = "/content/capstone-absa-colab", "unsloth/Llama-3.2-3B-Instruct", "/content/drive/MyDrive/absa-qlora-v4"
assert os.path.isdir(f"{PROJECT}/src"), "Project not found in /content: run the setup cell first (it unzips capstone-absa-colab-v4.zip)"
%cd $PROJECT
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
import shutil
shutil.copy("pip_freeze.txt", "results/pip_freeze.txt")
shutil.copytree("results", "/content/drive/MyDrive/absa-results-v4", dirs_exist_ok=True)
shutil.make_archive("/content/capstone-absa-colab-v4-results", "zip", "results")
from google.colab import files; files.download("/content/capstone-absa-colab-v4-results.zip")
print(open("results/comparison.md").read())

/content/capstone-absa-colab


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

| Method | Source test pair F1 (span) | Clothing pair F1 (category) | Clothing aspect F1 (category) | Clothing pair F1 (category, strict all reviews) | Clothing pair F1 (span, literal gold) | Abs. drop (span) | Rel. drop (span) | Net-sentiment corr. (Spearman) | Scored / gold reviews | JSON validity (clothing sentences) | s/review (clothing) |
|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.416 | 0.176 | 0.204 | 0.176 | 0.176 | 0.240 | 57.6% | -0.043 | 300 / 300 | 1.000 | 0.02 |
| baseline_lr | 0.384 | 0.158 | 0.204 | 0.158 | 0.160 | 0.224 | 58.3% | -0.153 | 300 / 300 | 1.000 | 0.02 |
| prompted | n/a | 0.291 | 0.365 | 0.278 | 0.195 | n/a | n/a | 0.718 | 225 / 300 | 0.926 | 16.10 |
| finetuned | 0.751 | 0.394 | 0.445 | 0.391 | 0.355 | 0.396 | 52.8% | 0.600 | 297 / 300 | 0.998 | 11.83 |

Gold aspects that occur literally in the review: 52.3% (whole word 39.9%). The span-level column can only score that share; the category-level columns score all gold aspects.



## Only if the diagnostic still shows the fine-tuned model returning `[]` for most clothing sentences
Retrain with fewer epochs: `!python -m src.finetune_qlora --model $MODEL --out $ADAPTER-v2 --epochs 2 --batch 4`, point `ADAPTER` at it and rerun the fine-tuned cells.